## Assumed task

Task from the IT-Jim Computer Vision course.

Detect the target object in the first frame using ORB keypoints matched with a marker image, then track it through the video with Lucas-Kanade optical flow and save the tracked result to an output video.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load marker and video
img = cv2.imread('../source/marker.jpg', cv2.IMREAD_GRAYSCALE)
cap = cv2.VideoCapture('../source/find_chocolate.mp4')

# Create ORB detector with 400 keypoints with a scaling pyramid factor of 1.2, nlevel
orb = cv2.ORB_create(1000, 1.2, 13)    

# Detect keypoints of original image
kp_mar, des_mar = orb.detectAndCompute(img, None)

# Retrieve first frame
ret, frame_old = cap.read()
# Get ORB for fist frame
frame_old = cv2.cvtColor(frame_old,cv2.COLOR_BGR2GRAY)
kp_old, des_old = orb.detectAndCompute(frame_old, None)

# Do matching 
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(des_mar,des_old)
  
# Select the matches based on distance
selected = []
for m in matches:
    if m.distance < 50:
        selected.append(m) 

# Number matches limit
if len(selected) > 10:
    # Extract the matched keypoints
#    src_pts = np.float32([ kp_mar[m.queryIdx].pt for m in selected ]).reshape(-1,1,2)
    p0 = np.float32([ kp_old[m.trainIdx].pt for m in selected ]).reshape(-1,1,2)

        
# Parameters for lucas kanade optical flow
lk_params = dict( winSize  = (15, 15),
                  maxLevel = 2,
                  criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# For video output
fps = 20
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
out = cv2.VideoWriter('ORB_Opt_Flow.avi',cv2.VideoWriter_fourcc('M','J','P','G'), 10, (frame_width,frame_height))

# Capture frame-by-frame
while True:      
    ret, frame_new = cap.read()
    if ret == True:
        frame_out = frame_new.copy()
        frame_new = cv2.cvtColor(frame_new,cv2.COLOR_BGR2GRAY)
        # calculate optical flow
        p1, st, err = cv2.calcOpticalFlowPyrLK(frame_old, frame_new, p0, None, **lk_params)

        # Select good points
        if p1 is not None:
            good_new = p1[st==1]
            good_old = p0[st==1]
        
        # draw rectangle
        x, y, w, h = cv2.boundingRect(good_new)
        frame_out = cv2.rectangle(frame_out, (x,y),(x + w, y + h),(0,255,0), 2)
        
        # Write video
        out.write(frame_out)
    
        # Now update the previous frame and previous points
        frame_old = frame_new.copy()
        p0 = good_new.reshape(-1, 1, 2)
    
    else:
        break

print("all done")

# Release all
cap.release()
out.release()
cv2.destroyAllWindows()    

all done
